# 1. Import librerías

In [1]:
import json

import soundfile as sf
import numpy as np
from scipy.signal import welch, butter, filtfilt, resample_poly
from scipy.signal.windows import hann
from plotly.express.colors import qualitative
import plotly.graph_objects as go
from plotly.subplots import make_subplots


colors = qualitative.Plotly

# 2. Cargar señal

In [2]:
file = "41223618_1.0_0_p4_3595"
wav_file = f"good_quality/wav/{file}.wav"
json_file = f"good_quality/json/{file}.json"

signal, fs = sf.read(wav_file)
t = np.arange(len(signal)) / fs

In [3]:
with open(json_file, "r") as f:
    data = json.load(f)
    events = data["event_annotation"]

onsets = [float(event["start"])/1000 for event in events]
offsets = [float(event["end"])/1000 for event in events]
annotations = [event["type"] for event in events]

# 3. Preprocesado

In [4]:
# Filtro Butterworth: 8º orden, pasa banda [70 - 1900] Hz
b, a = butter(4, [70, 1900], btype="bandpass", fs=fs)
signal_filtered = filtfilt(b, a, signal)

# Diezmado
fs_new = 4000
delta_fs = fs // fs_new
t_pre = t[::delta_fs]
signal_pre = resample_poly(signal_filtered, fs_new, fs)

# 4. Plot señal + Espectro

In [7]:
# Espectro PSD de Welch (original)
N = len(signal)//64
NFFT = 2**np.ceil(np.log2(N))
f, Pxx = welch(signal, fs=fs, window=hann(N), noverlap=N//2, nfft=NFFT)

# Espectro PSD Señal filtrada
N = len(signal_pre)//64
NFFT = 2**np.ceil(np.log2(N))
f_filtered, Pxx_filtered = welch(signal_pre, fs=fs_new, window=hann(N), noverlap=N//2, nfft=NFFT)

# Figura
fig = make_subplots(rows=2, cols=1,
    subplot_titles=("Respiratory Sound Signal", "Power Spectral Density (PSD)"),
)

# --- Señal (subplot 1) ---
fig.add_trace(
    go.Scatter(x=t, y=signal, mode="lines", name="Original Signal",
               line=dict(color=colors[0]), legend="legend"),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=t_pre, y=signal_pre, mode="lines", name="Preprocessed Signal",
               line=dict(color=colors[1]), legend="legend"),
    row=1, col=1
)

for onset, offset, annotation in zip(onsets, offsets, annotations):
    fig.add_vrect(x0=onset, x1=offset, fillcolor="red", opacity=0.25, line_width=1, line_color="black", row=1, col=1)
    fig.add_annotation(x=(onset + offset) / 2, y=0.97, xref="x1", yref="paper", text=annotation, showarrow=False, font=dict(size=12), opacity=0.8)

fig.update_xaxes(title="Time [s]", row=1, col=1)
fig.update_yaxes(title="Amplitude", row=1, col=1)

# --- PSD (subplot 2) ---
fig.add_trace(
    go.Scatter(x=f, y=10*np.log10(Pxx), mode="lines", name="PSD original",
               line=dict(color=colors[0]), legend="legend2"),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=f_filtered, y=10*np.log10(Pxx_filtered), mode="lines", name="PSD preprocessed",
               line=dict(color=colors[1]), legend="legend2"),
    row=2, col=1
)
fig.update_xaxes(title="Frequency [Hz]", row=2, col=1)
fig.update_yaxes(title="PSD [dB/Hz]", row=2, col=1)

fig.update_layout(
    height=600,
    width=1000,
    template="plotly_white",
    legend=dict(
        title="Signal",
        x=1.02, y=fig.layout.yaxis1.domain[1],
        xanchor="left", yanchor="top",
    ),
    legend2=dict(
        title="PSD",
        x=1.02, y=fig.layout.yaxis2.domain[1],
        xanchor="left", yanchor="top",
    ),
)

fig.show()